In [3]:
from boltzgen.data.parse.schema import YamlDesignParser
from pathlib import Path

MOL_DIR = "/home/anoroozi25/.cache/huggingface/hub/datasets--boltzgen--inference-data/snapshots/c3d36fd276e9caf098c75d4113c6d5eb320b1a4c/mols.zip"
yaml_path = Path("inputs/hallucination.yaml")

In [5]:
# Get itemn also needs to take a smaple id as input
parsed = YamlDesignParser(mol_dir=MOL_DIR).parse_yaml(yaml_path, mol_dir=MOL_DIR, mols={})
parsed

Target(record=Record(id='residue_index', structure=StructureInfo(resolution=None, method=None, deposited=None, released=None, revised=None, num_chains=1, num_interfaces=None, pH=None, temperature=None), chains=[ChainInfo(chain_id=0, chain_name=np.str_('A'), mol_type=0, cluster_id=-1, msa_id=-1, num_residues=27, valid=True, entity_id=0)], interfaces=[], templates=None), structure=Structure(atoms=array([('N', [0., 0., 0.],  True, 0., 1.),
       ('CA', [0., 0., 0.],  True, 0., 1.),
       ('C', [0., 0., 0.],  True, 0., 1.),
       ('O', [0., 0., 0.],  True, 0., 1.),
       ('N', [0., 0., 0.],  True, 0., 1.),
       ('CA', [0., 0., 0.],  True, 0., 1.),
       ('C', [0., 0., 0.],  True, 0., 1.),
       ('O', [0., 0., 0.],  True, 0., 1.),
       ('N', [0., 0., 0.],  True, 0., 1.),
       ('CA', [0., 0., 0.],  True, 0., 1.),
       ('C', [0., 0., 0.],  True, 0., 1.),
       ('O', [0., 0., 0.],  True, 0., 1.),
       ('N', [0., 0., 0.],  True, 0., 1.),
       ('CA', [0., 0., 0.],  True, 0., 1

In [6]:
structure = parsed.structure
design_info = parsed.design_info

In [ ]:
# Tokenize structure
import numpy as np
from boltzgen.data.tokenize.tokenizer import Tokenizer

tokenized = Tokenizer().tokenize(structure)

# Transfer conditioning information that is stored in tokens
token_to_res = tokenized.token_to_res
tokenized.tokens["design_mask"] = design_info.res_design_mask[token_to_res]
tokenized.tokens["binding_type"] = design_info.res_binding_type[token_to_res]
tokenized.tokens["structure_group"] = design_info.res_structure_groups[token_to_res] 

# Propagate design mask to obtain chain_design_mask (True whenever something is covalently bound to any residue that is in a chain that contains a design residue).
chain_design_mask = tokenized.tokens["design_mask"].astype(bool)
asym_id = tokenized.tokens["asym_id"]
while True:
    design_chains = np.unique(asym_id[chain_design_mask])
    chain_propagated = np.isin(asym_id, design_chains)
    for i, j, _ in tokenized.bonds:
        if any([chain_propagated[i], chain_propagated[j]]):
            chain_propagated[i] = True
            chain_propagated[j] = True
    if np.equal(chain_propagated, chain_design_mask).all():
        break
    chain_design_mask = chain_propagated.astype(bool)

# Try to find molecules in the dataset moldir if provided
# Find missing ones in global moldir and check if all found
molecules = {}
molecules.update(self.canonicals)
mol_names = set(tokenized.tokens["res_name"].tolist())
mol_names = mol_names - set(self.canonicals.keys())
mol_names = mol_names - set(parsed.extra_mols.keys())
if self.moldir is not None:
    molecules.update(load_molecules(self.moldir, mol_names))

mol_names = mol_names - set(molecules.keys())
molecules.update(load_molecules(self.moldir, mol_names))
molecules.update(parsed.extra_mols)

# Finalize input data
input_data = Input(
    tokens=tokenized.tokens,
    bonds=tokenized.bonds,
    token_to_res=token_to_res,
    structure=structure,
    msa={},
    templates=None,
)

# Compute features
features = self.dataset.featurizer.process(
    input_data,
    molecules=molecules,
    random=np.random.default_rng(None),
    training=False,
    max_seqs=1,
    backbone_only=self.backbone_only,
    atom14=self.atom14,
    atom37=self.atom37,
    design=self.design,
    override_method="X-RAY DIFFRACTION",
    compute_affinity=self.compute_affinity,
    disulfide_prob=self.disulfide_prob,
    disulfide_on=self.disulfide_on,
)

# transfer secondary structure conditioning
ss_type = design_info.res_ss_types[token_to_res]
features["ss_type"] = torch.from_numpy(ss_type).to(features["ss_type"])
features["design_ss_mask"][ss_type != const.ss_type_ids["UNSPECIFIED"]] = 1

# set chain_design_mask
features["chain_design_mask"] = torch.from_numpy(chain_design_mask)

# Compute template features
templates_features = load_dummy_templates(
    tdim=1, num_tokens=len(features["res_type"])
)
features.update(templates_features)

# set last necessary features
features["idx_dataset"] = torch.tensor(1)

# If a smaple id is provided then this should be the sample id instead of path.stem
if sample_id is not None:
    features["id"] = sample_id
else:
    features["id"] = path.stem
if "structure" in self.extra_features:
    features["structure"] = structure
if "tokenized" in self.extra_features:
    features["tokenized"] = tokenized

return features